In [0]:
# ============================================================
#  RAW → SILVER: VENTAS
# ============================================================

STORAGE_ACCOUNT  = "stdatacolake"
CONTAINER_RAW    = "raw-zone"
CONTAINER_SILVER = "silver-zone"

RAW_PATH    = f"abfss://{CONTAINER_RAW}@{STORAGE_ACCOUNT}.dfs.core.windows.net/SAP"
SILVER_PATH = f"abfss://{CONTAINER_SILVER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/ventas"

In [0]:
# ── LEER CSV DESDE RAW ────────────────────────────────────────
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, TimestampType

schema_ventas = StructType([
    StructField("id_visita",          IntegerType(),  nullable=False),
    StructField("id_vendedor",        IntegerType(),  nullable=True),
    StructField("fecha_visita",       TimestampType(),nullable=True),
    StructField("tipo_visita",        StringType(),   nullable=True),
    StructField("resultado",          StringType(),   nullable=True),
    StructField("valor_cartera",      DoubleType(),   nullable=True),
    StructField("acuerdo_comercial",  StringType(),   nullable=True),
])

df_raw = spark.read \
    .schema(schema_ventas) \
    .option("header", "true") \
    .csv(f"{RAW_PATH}/1_SAP_fact_ventas.csv")

print(f"✅ Raw leído: {df_raw.count()} filas")
df_raw.show(5)

In [0]:
# ── LIMPIEZA Y TRANSFORMACIONES ───────────────────────────────
from pyspark.sql.functions import col, trim, when, current_timestamp

df_silver = df_raw \
    .filter(col("id_visita").isNotNull()) \
    .filter(col("valor_cartera") > 0) \
    .withColumn("tipo_visita",       trim(col("tipo_visita"))) \
    .withColumn("resultado",         trim(col("resultado"))) \
    .withColumn("acuerdo_comercial", trim(col("acuerdo_comercial"))) \
    .withColumn("resultado_flag",
        when(col("resultado") == "Acuerdo",    "GANADO")
       .when(col("resultado") == "Pendiente",  "EN_PROCESO")
       .otherwise("PERDIDO")
    ) \
    .dropDuplicates(["id_visita"]) \
    .withColumn("_updated_at", current_timestamp())

print(f"✅ Silver listo: {df_silver.count()} filas")
df_silver.show(5)

In [0]:
# ── GUARDAR COMO DELTA EN SILVER ─────────────────────────────
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .save(SILVER_PATH)

print(f"✅ Guardado en Silver: {SILVER_PATH}")